In [ ]:
%load_ext autoreload
%autoreload 

import pandas as pd
import os
import numpy as np
from os.path  import join
from sklearn.model_selection import train_test_split
import torch
from PIL import Image
from tqdm import tqdm
from torchvision import transforms

from src.utils.data_utils import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Paths

In [4]:
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# =============================================================
# CONFIGURATION — set these paths before running
# =============================================================
# Root directory of this repository
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))

# ---- Raw data locations (download instructions in README) ----
# Raw MNIST processed tensors (training.pt, test.pt)
MNIST_RAW_DATA_FOLDER = '/path/to/MNIST/processed'

# MIMIC-CXR-JPG root (physionet download)
MIMIC_ROOT_DATA_FOLDER = '/path/to/mimic-cxr-jpg/2.0.0'
MIMIC_IMAGE_PREFIX_PATH = os.path.join(MIMIC_ROOT_DATA_FOLDER, 'files')

# MIMIC-IV hospital tables (patients.csv.gz, admissions.csv.gz)
MIMIC_LOCAL_DATA_FOLDER = '/path/to/mimiciv/2.2/hosp'

# HAM10000 metadata CSVs
HAM_METADATA_PATH = os.path.join(BASE_DIR, 'data/HAM10000/HAM10000_metadata.csv')
ISIC_METADATA_PATH = os.path.join(BASE_DIR, 'data/HAM10000/ISIC2018_Task3_Test_GroundTruth.csv')


In [14]:
cols_of_interest_dict = {}
cols_of_interest_dict['mnist'] = ['A','S', 'Random_binary']
cols_of_interest_dict['mimic'] = ['ViewPosition_binary','PatientOrientationCodeSequence_CodeMeaning_binary','Support_Devices_binary','Gender_binary','Insurance_binary','Language_binary','Marital_Status_binary','Race_cat_binary','Age_binary', 'PerformedProcedureStepDescription_binary','Random_binary']
cols_of_interest_dict['ham'] = ['Sex_binary', 'Age_binary','Dataset_binary', 'Localization_binary','Random_binary']
cols_of_interest_dict['civil_comments'] = ['Gender_binary','Orientation_binary','Religion_binary','Race_binary', 'Year_binary', 'Random_binary']


# MNIST

## Process data and make subgroups

In [7]:
raw_data_folder = MNIST_RAW_DATA_FOLDER

train_data = torch.load(os.path.join(raw_data_folder, 'training.pt'))
test_data = torch.load(os.path.join(raw_data_folder, 'test.pt'))

# Separate images and labels
train_images, train_labels = train_data
test_images, test_labels = test_data

# MAKE DFs
train_val_images_df = pd.DataFrame()
index_array = np.linspace(0,len(train_images)-1,len(train_images),dtype=int)
train_val_images_df['image_index'] = index_array
train_val_images_df['image_label'] = train_labels
train_val_images_df['Y'] = train_val_images_df['image_label']%2 # 0 if even, 1 if odd
train_val_images_df['S'] = np.random.choice([0, 1], size=len(train_val_images_df))
train_val_images_df['A'] = train_val_images_df['image_label'].apply(lambda x: 0 if x < 5 else 1)
train_val_images_df['Random_binary'] = np.random.choice([0, 1], size=len(train_val_images_df))

test_images_df = pd.DataFrame()
index_array = np.linspace(0,len(test_images)-1,len(test_images),dtype=int)
test_images_df['image_index'] = index_array
test_images_df['image_label'] = test_labels
test_images_df['Y'] = test_images_df['image_label']%2 # 0 if even, 1 if odd
test_images_df['S'] = np.random.choice([0, 1], size=len(test_images_df))
test_images_df['A'] = test_images_df['image_label'].apply(lambda x: 0 if x < 5 else 1)
test_images_df['Random_binary'] = np.random.choice([0, 1], size=len(test_images_df))

In [8]:
subset_train_val_images = make_subset_images(train_val_images_df,train_images,fg_colour_channel_col='S')
subset_test_images = make_subset_images(test_images_df,test_images,fg_colour_channel_col='S')

save_data_folder = 'data/MNIST/parity'
torch.save(subset_train_val_images.int(), os.path.join(save_data_folder,'train_val_images.pt'))
torch.save(subset_test_images.int(), os.path.join(save_data_folder, 'test_images.pt'))

val_images_df = train_val_images_df.iloc[int(len(train_val_images_df)*0.8):]
train_images_df = train_val_images_df.iloc[:int(len(train_val_images_df)*0.8)]

train_images_df.to_csv(os.path.join(save_data_folder, 'train_labels.csv'))
val_images_df.to_csv(os.path.join(save_data_folder, 'val_labels.csv'))
test_images_df.to_csv(os.path.join(save_data_folder, 'test_labels.csv'))

## Make pre-training datasets

In [9]:
test_df = pd.read_csv('data/MNIST/parity/test_labels.csv')
train_val_df, test_df = train_test_split(test_df, test_size=0.4,random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=1/6, random_state=42)

In [ ]:
train_df.to_csv('data/MNIST/parity/train_labels_pretrain.csv',index=False)
val_df.to_csv('data/MNIST/parity/val_labels_pretrain.csv',index=False)
test_df.to_csv('data/MNIST/parity/test_labels_pretrain.csv',index=False)

## Make alloc datasets (fine-tuning)

In [ ]:
label = 'parity'
df_paths = [
    f'data/MNIST/{label}/train_labels.csv',
    f'data/MNIST/{label}/val_labels.csv',
    f'data/MNIST/{label}/test_labels.csv'
]

cols_of_interest = cols_of_interest_dict['mnist']

min_group_size_train = float('inf')
min_group_size_val = float('inf')

for col in cols_of_interest:
    for path in df_paths:
        df = pd.read_csv(path)
        group_0_size = df[df[col] == 0].shape[0]
        group_1_size = df[df[col] == 1].shape[0]

        min_group_size = min(group_0_size,group_1_size)

        if 'train' in path:
            if min_group_size < min_group_size_train:
                min_group_size_train = min_group_size
        elif 'val' in path:
            if min_group_size < min_group_size_val:
                min_group_size_val = min_group_size


In [ ]:
train_size = min_group_size_train
val_size = min_group_size_val

train_df = pd.read_csv(f'data/MNIST/{label}/train_labels.csv')
val_df = pd.read_csv(f'data/MNIST/{label}/val_labels.csv')

output_dir = f"data/MNIST/{label}"

for seed in [42,43,44]:
    for col in cols_of_interest:
        os.makedirs(os.path.join(output_dir, col), exist_ok=True)

        train_subsets = make_proportion_dfs(train_df, col, train_size, seed)
        val_subsets = make_proportion_dfs(val_df, col, val_size, seed)

        for pct, df in train_subsets:
            filename = f"train_{pct}_{seed}.csv"
            path = os.path.join(output_dir, col, filename)
            df.to_csv(path, index=False)

        for pct, df in val_subsets:
            filename = f"val_{pct}_{seed}.csv"
            path = os.path.join(output_dir, col, filename)
            df.to_csv(path, index=False)

## Make baseline datasets
Datasets of same size as alloc datasets, useful for, eg, hyperparam tuning

In [ ]:
for seed in [42,43,44]:

    train_df = pd.read_csv(f'data/MNIST/{label}/train_labels.csv')
    val_df = pd.read_csv(f'data/MNIST/{label}/val_labels.csv')

    subset_train_df = train_df.sample(n=min_group_size_train, replace=False, random_state=seed)
    subset_val_df = val_df.sample(n=min_group_size_val, replace=False, random_state=seed)

    subset_train_df.to_csv(f'data/MNIST/{label}/train_labels_{min_group_size_train}_{seed}.csv', index=False)
    subset_val_df.to_csv(f'data/MNIST/{label}/val_labels_{min_group_size_val}_{seed}.csv', index=False)

# MIMIC CXR

## Process data and make subgroups

In [12]:
def find_images(folder_path):
    image_paths = []

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith('.jpg'):
                image_paths.append(os.path.join(root, file))

    return image_paths

In [13]:
label = 'Pleural_Effusion'

In [ ]:
patient_categories = [x for x in os.listdir(os.path.join(MIMIC_ROOT_DATA_FOLDER,'files')) if 'index' not in x] #['p16', 'p17', 'p11', 'p18', 'p12', 'p13', 'p14', 'p10', 'p19', 'p15']

all_patient_paths=[]
for patient_cat in patient_categories:
    patient_path = os.path.join(os.path.join(MIMIC_ROOT_DATA_FOLDER,'files'),patient_cat)
    all_patient_paths.append(find_images(patient_path))

flattened_list = [item for sublist in all_patient_paths for item in sublist]

image_paths_df = pd.DataFrame()
image_paths_df['path']=flattened_list

image_paths_df['patient_cat'] = image_paths_df['path'].apply(lambda x: x.split("/")[-4])
image_paths_df['subject_id'] = image_paths_df['path'].apply(lambda x: x.split("/")[-3])
image_paths_df['subject_id'] = image_paths_df['subject_id'].apply(lambda x: int(x.replace("p","")))
image_paths_df['study_id'] = image_paths_df['path'].apply(lambda x: x.split("/")[-2])
image_paths_df['study_id'] = image_paths_df['study_id'].apply(lambda x: int(x.replace("s","")))
image_paths_df['dicom_id'] = image_paths_df['path'].apply(lambda x: x.split("/")[-1].split(".")[0])

In [ ]:
image_metadata_path = os.path.join(MIMIC_ROOT_DATA_FOLDER, 'mimic-cxr-2.0.0-metadata.csv.gz')
image_metadata_df = pd.read_csv(image_metadata_path)
image_metadata_df.drop(columns=['Rows','Columns','StudyTime','ProcedureCodeSequence_CodeMeaning','ViewCodeSequence_CodeMeaning'], inplace=True)
full_metadata_df = pd.merge(image_paths_df, image_metadata_df, on = ['subject_id','study_id','dicom_id'], how='left')

In [ ]:
chexpert_path = os.path.join(MIMIC_ROOT_DATA_FOLDER, 'mimic-cxr-2.0.0-chexpert.csv.gz')
chexpert_df = pd.read_csv(chexpert_path)
full_metadata_df = pd.merge(full_metadata_df, chexpert_df, on = ['subject_id','study_id'], how='inner') # this removes 13 rows for which no annotations have been made (apparently)

In [ ]:
patients_file = 'patients.csv.gz'
admissions_file = 'admissions.csv.gz'

patients_df = pd.read_csv(os.path.join(MIMIC_LOCAL_DATA_FOLDER,patients_file))
patients_df.drop(columns=['anchor_year','anchor_year_group','dod'], inplace=True)
full_metadata_df = pd.merge(full_metadata_df, patients_df, on = ['subject_id'], how='left')

In [ ]:
admissions_df = pd.read_csv(os.path.join(MIMIC_LOCAL_DATA_FOLDER,admissions_file))
admissions_df = admissions_df[['subject_id','hadm_id','admittime','dischtime','insurance','language','marital_status','race']] # keep only relevant columns
admissions_df['admittime'] = admissions_df['admittime'].apply(lambda x: str(x)[:10].replace("-", ""))
admissions_df['dischtime'] = admissions_df['dischtime'].apply(lambda x: str(x)[:10].replace("-", ""))

hadm_ids = []

for index, row in full_metadata_df.iterrows():
    subject_id = row['subject_id']
    subject_admissions = admissions_df[admissions_df['subject_id'] == subject_id]
    study_date = row['StudyDate']

    hadm_id = None

    for index, admission_row in subject_admissions.iterrows():
        admission_date = int(admission_row['admittime'])

        if study_date >= admission_date:
            if study_date <= int(admission_row['dischtime']):
                hadm_id = admission_row['hadm_id']

    if hadm_id is None:
        if len(subject_admissions) == 0:
            hadm_id = 0 # just return 0, should give Nans for the other values in the merge
        else:
            hadm_id = subject_admissions.iloc[0]['hadm_id'].item() # just return the first one from the same subject
    
    hadm_ids.append(hadm_id)

full_metadata_df['hadm_id'] = hadm_ids

full_metadata_df = pd.merge(full_metadata_df, admissions_df, on = ['subject_id','hadm_id'], how='left')
full_metadata_df.drop(columns=['hadm_id'], inplace=True)

In [ ]:
full_metadata_df['Y'] = full_metadata_df[label].apply(lambda x: 1 if x == 1 else 0)
full_metadata_df = full_metadata_df[full_metadata_df[label] != -1] # delete any uncertain rows
filtered_df = full_metadata_df.dropna(subset=['gender','anchor_age','race','insurance']) # deletes about 44520 rows

In [ ]:
# binarise / categorise some cols
filtered_df['Race_cat'] = filtered_df['race'].apply(lambda x: 'White' if x=='WHITE' else 
                                       'White-other' if 'WHITE -' in x or 'PORTUGUESE' in x else 
                                       'Hispanic' if 'HISPANIC' in x or 'LATINO' in x or 'MEXICAN' in x or 'CENTRAL AMERICAN' in x or 'CUBAN' in x or 'SALVADORAN' in x or 'GUATEMALAN' in x or 'DOMINICAN' in x or 'COLOMBIAN' in x or 'HONDURAN' in x or 'SOUTH AMERICAN' in x else 
                                       'Asian' if 'ASIAN' in x or 'CHINESE' in x or 'KOREAN' in x or 'SOUTH EAST ASIAN' in x or 'ASIAN INDIAN' in x else 
                                       'Black' if 'BLACK' in x or 'AFRICAN AMERICAN' in x or 'CARIBBEAN ISLAND' in x or 'CAPE VERDEAN' in x else 
                                       'Other')
filtered_df['Age_multi'] = filtered_df['anchor_age'].values.astype('int')
filtered_df['Age_multi'] = np.where(filtered_df['Age_multi'].between(-1,29), 0, filtered_df['Age_multi'])
filtered_df['Age_multi'] = np.where(filtered_df['Age_multi'].between(30,49), 1, filtered_df['Age_multi'])
filtered_df['Age_multi'] = np.where(filtered_df['Age_multi'].between(50,69), 2, filtered_df['Age_multi'])
filtered_df['Age_multi'] = np.where(filtered_df['Age_multi']>=70, 3, filtered_df['Age_multi'])

filtered_df['Age_binary'] = filtered_df['anchor_age'].values.astype('int')
filtered_df['Age_binary'] = np.where(filtered_df['Age_binary'].between(-1, 60), 0, filtered_df['Age_binary'])
filtered_df['Age_binary'] = np.where(filtered_df['Age_binary']>= 60, 1, filtered_df['Age_binary'])

filtered_df['ViewPosition_binary'] = np.where(
        filtered_df['ViewPosition'].isin(['AP', 'PA']),
        1,
        np.where(filtered_df['ViewPosition'].isna(), np.nan, 0)
    )

filtered_df['PatientOrientationCodeSequence_CodeMeaning_binary'] = np.where(
        filtered_df['PatientOrientationCodeSequence_CodeMeaning'] == 'Erect',
        1,
        np.where(filtered_df['PatientOrientationCodeSequence_CodeMeaning'].isna(), np.nan, 0)
    )

filtered_df['Support_Devices_binary'] = (filtered_df['Support Devices'] == 1.0).astype(int)

filtered_df['Gender_binary'] = (filtered_df['gender'] == 'M').astype(int)

filtered_df['Insurance_binary'] = (filtered_df['insurance'] == 'Private').astype(int)

filtered_df['Language_binary'] = np.where(
        filtered_df['language'] == 'English',
        1,
        np.where(filtered_df['language'].isna(), np.nan, 0)
    )

filtered_df['Marital_Status_binary'] = (filtered_df['marital_status'] == 'MARRIED').astype(int)

filtered_df['Race_cat_binary'] = filtered_df['Race_cat'].isin(['White', 'White-other']).astype(int)
filtered_df['PerformedProcedureStepDescription_binary'] = np.where(
        filtered_df['PerformedProcedureStepDescription'] == 'CHEST (PA AND LAT)',
        1,
        np.where(
            filtered_df['PerformedProcedureStepDescription'].isin(['CHEST (PORTABLE AP)', 'PORTABLE ABDOMEN', 'CHEST (PA AND LAT) PORT']),
            0,
            np.nan
        ))

filtered_df['Random_binary'] = np.random.randint(0, 2, size=len(filtered_df))

In [ ]:
def split(all_meta, patient_ids, fraction_test = 0.33, fraction_test_valtest= 0.66):
    sub_train, sub_val_test = train_test_split(patient_ids, test_size=fraction_test, random_state=0) # splits based on patient_id (not studies)
    sub_val, sub_test = train_test_split(sub_val_test, test_size=fraction_test_valtest, random_state=0)
    train_meta = all_meta[all_meta.subject_id.isin(sub_train)]
    val_meta = all_meta[all_meta.subject_id.isin(sub_val)]
    test_meta = all_meta[all_meta.subject_id.isin(sub_test)]
    return train_meta, val_meta, test_meta

sub_train, sub_val, sub_test = split(filtered_df, np.unique(filtered_df['subject_id']), fraction_test = 0.33, fraction_test_valtest = 0.66)

sub_train.to_csv(f'data/MIMIC/{label}/train_labels.csv.gz', index=False)
sub_val.to_csv(f'data/MIMIC/{label}/val_labels.csv.gz', index=False)
sub_test.to_csv(f'data/MIMIC/{label}/test_labels.csv.gz', index=False)

## Resave smaller images

In [ ]:
prefix = MIMIC_IMAGE_PREFIX_PATH
output_base = os.path.join(BASE_DIR, "data/MIMIC/images")

os.makedirs(output_base, exist_ok=True)

transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.PILToTensor(),  # Returns [1, H, W] for grayscale
        transforms.ConvertImageDtype(torch.float32)
        ])

for df_path in ['data/MIMIC/Pleural_Effusion/train_labels.csv.gz', 'data/MIMIC/Pleural_Effusion/val_labels.csv.gz', 'data/MIMIC/Pleural_Effusion/test_labels.csv.gz']:

    df = pd.read_csv(df_path)
    new_paths = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_path = row["path"]
        rel_path = os.path.relpath(img_path, prefix)  # gives pxx/.../filename.jpg
        rel_path_pt = os.path.splitext(rel_path)[0] + ".pt"

        output_path = os.path.join(output_base, rel_path_pt)

        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        if not os.path.exists(output_path):
            img = Image.open(img_path).convert("L")
            tensor = transform(img)  # [1, 256, 256]
            torch.save(tensor, output_path)
        new_paths.append(output_path)

    df["tensor_path"] = new_paths
    df.to_csv(df_path, index=False)

## Make pre-training datasets

In [ ]:
test_df = pd.read_csv('data/MIMIC/Pleural_Effusion/test_labels.csv.gz')
train_val_df, test_df = train_test_split(test_df, test_size=0.4,random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=1/6, random_state=42)

In [ ]:
train_df.to_csv('data/MIMIC/Pleural_Effusion/train_labels_pretrain.csv.gz',index=False)
val_df.to_csv('data/MIMIC/Pleural_Effusion/val_labels_pretrain.csv.gz',index=False)
test_df.to_csv('data/MIMIC/Pleural_Effusion/test_labels_pretrain.csv.gz',index=False)

## Make alloc datasets for fine-tuning

In [ ]:
df_paths = [
    f'data/MIMIC/{label}/train_labels.csv.gz',
    f'data/MIMIC/{label}/val_labels.csv.gz',
    f'data/MIMIC/{label}/test_labels.csv.gz'
]

cols_of_interest = cols_of_interest_dict['mimic']

min_group_0_size_train = float('inf')
min_group_0_size_val = float('inf')

for col in cols_of_interest:
    for path in df_paths:
        df = pd.read_csv(path)
        print(df[col].value_counts(dropna=False))
        group_0_size = df[df[col] == 0].shape[0]

        if 'train' in path:
            if group_0_size < min_group_0_size_train:
                min_group_0_size_train = group_0_size
        elif 'val' in path:
            if group_0_size < min_group_0_size_val:
                min_group_0_size_val = group_0_size

In [ ]:
train_df = pd.read_csv(f'data/MIMIC/{label}/train_labels.csv.gz')
val_df = pd.read_csv(f'data/MIMIC/{label}/val_labels.csv.gz')

train_size = min_group_0_size_train
val_size = min_group_0_size_val

output_dir = f"data/MIMIC/{label}_{train_size}"

for seed in [42,43,44]:

    for col in cols_of_interest:
        os.makedirs(os.path.join(output_dir, col), exist_ok=True)

        train_subsets = make_proportion_dfs(train_df, col, train_size, seed)
        val_subsets = make_proportion_dfs(val_df, col, val_size, seed)

        for pct, df in train_subsets:
            filename = f"train_{pct}_{seed}.csv"
            path = os.path.join(output_dir, col, filename)
            df.to_csv(path, index=False)

        for pct, df in val_subsets:
            filename = f"val_{pct}_{seed}.csv"
            path = os.path.join(output_dir, col, filename)
            df.to_csv(path, index=False)

## Make baseline datasets 

In [ ]:
for seed in [42,43,44]:

    train_df = pd.read_csv(f'data/MIMIC/{label}/train_labels.csv.gz')
    val_df = pd.read_csv(f'data/MIMIC/{label}/val_labels.csv.gz')

    subset_train_df = train_df.sample(n=min_group_0_size_train, replace=False, random_state=seed)
    subset_val_df = val_df.sample(n=min_group_0_size_val, replace=False, random_state=seed)

    subset_train_df.to_csv(f'data/MIMIC/{label}/train_labels_{min_group_0_size_train}_{seed}.csv.gz', index=False)
    subset_val_df.to_csv(f'data/MIMIC/{label}/val_labels_{min_group_0_size_val}_{seed}.csv.gz', index=False)

# HAM10000

## Process data

In [ ]:
metadata_df = pd.read_csv(HAM_METADATA_PATH)
isic_metadata_df = pd.read_csv(ISIC_METADATA_PATH)
full_metadata_df = pd.concat([metadata_df,isic_metadata_df])

In [ ]:
extremities = {
    "lower extremity", "upper extremity", "foot", "hand", "acral", "unknown"
}
full_metadata_df['Path'] = full_metadata_df['image_id'].apply(lambda x: 'data/HAM10000/images/'+x+'.jpg' )
full_metadata_df = full_metadata_df[full_metadata_df["Path"].apply(os.path.exists)]

full_metadata_df['Y'] = full_metadata_df['dx'].apply(lambda x: 1 if x=='mel' or x == 'akiec' else 0)
full_metadata_df['Sex_binary'] = full_metadata_df['sex'].apply(lambda x: 1 if x=='male' else 0 if x=='female' else np.nan)
full_metadata_df['Age_binary'] = full_metadata_df['age'].apply(lambda x: 1 if x>50 else 0 if x<=50 else np.nan)
full_metadata_df['Dataset_binary'] = full_metadata_df['dataset'].apply(lambda x: 1 if x=='vidir_molemax' or x=='rosendahl' else 0)
full_metadata_df["Localization_binary"] = full_metadata_df["localization"].apply(lambda x: "extremity" if x in extremities else "central")
full_metadata_df["Localization_binary"] = full_metadata_df["Localization_binary"].apply(lambda x:1 if x=="extremity" else 0)
full_metadata_df['Random_binary'] = np.random.randint(0, 2, size=len(full_metadata_df))
full_metadata_df.to_csv('data/HAM10000/processed_metadata.csv')

In [ ]:
nan_rows = full_metadata_df[full_metadata_df.isna().any(axis=1)]
non_nan_rows = full_metadata_df.drop(nan_rows.index)

n_needed = 2000 - len(nan_rows)
sampled_non_nan = non_nan_rows.sample(n=n_needed, random_state=42)

pretraining_df = pd.concat([nan_rows, sampled_non_nan])
finetuning_df = full_metadata_df.drop(pretraining_df.index)

## Pre-training dfs

In [ ]:
pretrain_train_df, pretrain_val_df = train_test_split(pretraining_df, test_size=0.2,random_state=42)
finetune_train_val_df, test_df = train_test_split(finetuning_df, test_size=0.11, random_state=42)

pretrain_train_df.to_csv('data/HAM10000/train_labels_pretrain.csv',index=False)
pretrain_val_df.to_csv('data/HAM10000/val_labels_pretrain.csv',index=False)
test_df.to_csv('data/HAM10000/test_labels.csv',index=False)

## Make alloc datasets (fine-tuning)

In [ ]:
cols_of_interest = cols_of_interest_dict['ham']
min_group_0_size = float('inf')
min_group_1_size = float('inf')

for col in cols_of_interest:
    print(finetune_train_val_df[col].value_counts(dropna=False))
    group_0_size = finetune_train_val_df[finetune_train_val_df[col] == 0].shape[0]
    if group_0_size < min_group_0_size:
        min_group_0_size = group_0_size
    group_1_size = finetune_train_val_df[finetune_train_val_df[col] == 1].shape[0]
    if group_1_size < min_group_1_size:
        min_group_1_size = group_1_size

min_group_size = min(min_group_0_size,min_group_1_size)
print(min_group_size)

In [ ]:
train_val_size = min_group_size

output_dir = "data/HAM10000"

for seed in [42,43,44]:
    for col in cols_of_interest:
        os.makedirs(os.path.join(output_dir, col), exist_ok=True)

        train_val_subsets = make_proportion_dfs(finetune_train_val_df, col, train_val_size, seed)

        for pct, df in train_val_subsets:
            train_df,val_df = train_test_split(df, test_size=0.15, random_state=seed)

            train_filename = f"train_{pct}_{seed}.csv"
            train_path = os.path.join(output_dir, col, train_filename)
            train_df.to_csv(train_path, index=False)

            val_filename = f"val_{pct}_{seed}.csv"
            val_path = os.path.join(output_dir, col, val_filename)
            val_df.to_csv(val_path, index=False)

# Civil_Comments

## Process data

In [ ]:
civil_comments_dir = 'data/CIVILCOMMENTS'

annot_data_df = load_df(os.path.join(civil_comments_dir))
processed_annot_data_df = augment_df(annot_data_df)

processed_annot_data_df['year']=processed_annot_data_df['created_date'].apply(lambda x: x[0:4])
processed_annot_data_df['Year_binary'] = processed_annot_data_df['year'].apply(lambda x: 1 if int(x)>=2017 else 0)
processed_annot_data_df['Random_binary'] = np.random.randint(0, 2, size=len(processed_annot_data_df))
processed_annot_data_df['Y'] = (processed_annot_data_df['toxicity'] >= 0.5).astype(int)

processed_annot_data_df = processed_annot_data_df[processed_annot_data_df['comment_text'].apply(lambda x: isinstance(x, str))]

In [ ]:
to_rename = ['gender_any','orientation_any','religion_any','race_any']
processed_annot_data_df = processed_annot_data_df.rename(
    columns={col: col.split("_")[0].capitalize() + "_binary" for col in to_rename}
)

## Make pre-training datasets

In [ ]:
train_df,test_df = train_test_split(processed_annot_data_df, test_size=0.05, random_state=42)
pretrain_train_val_df, finetune_train_val_df = train_test_split(processed_annot_data_df, test_size=0.96,random_state=42)
pretrain_train_df, pretrain_val_df = train_test_split(pretrain_train_val_df, test_size=0.2,random_state=42)

pretrain_train_df.to_csv('data/CIVILCOMMENTS/train_labels_pretrain.csv',index=False)
pretrain_val_df.to_csv('data/CIVILCOMMENTS/val_labels_pretrain.csv',index=False)
test_df.to_csv('data/CIVILCOMMENTS/test_labels.csv',index=False)

## Make alloc datasets (fine-tuning)

In [ ]:
cols_of_interest = cols_of_interest['civil_comments']

min_group_0_size = float('inf')
min_group_1_size = float('inf')

for col in cols_of_interest:
    print(finetune_train_val_df[col].value_counts(dropna=False))
    group_0_size = finetune_train_val_df[finetune_train_val_df[col] == 0].shape[0]
    if group_0_size < min_group_0_size:
        min_group_0_size = group_0_size
    group_1_size = finetune_train_val_df[finetune_train_val_df[col] == 1].shape[0]
    if group_1_size < min_group_1_size:
        min_group_1_size = group_1_size

min_group_size = min(min_group_0_size,min_group_1_size)

In [ ]:
train_val_size = min_group_size

output_dir = "data/CIVILCOMMENTS"

for seed in [42,43,44]:

    for col in cols_of_interest:
        os.makedirs(os.path.join(output_dir, col), exist_ok=True)

        train_val_subsets = make_proportion_dfs(finetune_train_val_df, col, train_val_size, seed)

        for pct, df in train_val_subsets:
            train_df,val_df = train_test_split(df, test_size=0.15, random_state=seed)

            train_filename = f"train_{pct}_{seed}.csv"
            train_path = os.path.join(output_dir, col, train_filename)
            train_df.to_csv(train_path, index=False)

            val_filename = f"val_{pct}_{seed}.csv"
            val_path = os.path.join(output_dir, col, val_filename)
            val_df.to_csv(val_path, index=False)


# FMMIMIC

See `scripts/get_chexagent_embeds.py` and `scripts/get_maira_embeds.py` to get the respective foundation model embeddings and use them for fine-tuning.